# End-to-end: Harmonic Oscillator → Train Action-Angle Network

このノートでは、既存の実装をなるべく再利用しつつ、
(1) 調和振動のシミュレーション生成 → (2) 学習 → (3) 可視化/評価 までを一通り実行します。

モデルは Action-Angle Network (AAN) を使用します。エンコーダ/デコーダはフロー型 (shear) を小さめの層数で構成し、
短時間で動作確認できるように軽量設定にしています。

In [ ]:
import os, shutil, pathlib, json
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from absl import logging
import ml_collections

from action_angle_networks import train, analysis
from action_angle_networks.simulation import harmonic_motion_simulation as hsim
from action_angle_networks.configs.harmonic_motion import default as hm_default

%config InlineBackend.figure_format = 'retina'
logging.set_verbosity(logging.INFO)

## 1) 設定とデータ生成
- 調和振動のシミュレーションパラメータをサンプリング
- 正準座標 (q, p) を生成
- 可視化（位相空間）

In [ ]:
# ベース設定をロード
config = hm_default.get_config()

# デモ用に軽量設定へ上書き
config = config.copy_and_resolve_references()
config.rng_seed = 0
config.num_trajectories = 2              # 2 DOF (2本の1D振動子)
config.dimensions_per_trajectory = 1
config.num_samples = 600                 # サンプル数
config.time_delta = 0.1                  # サンプリング間隔
config.train_time_jump_schedule = 'linear'
config.train_time_jump_range = (1, 10)
config.test_time_jumps = (1, 2, 5, 10, 20)
config.train_split_proportion = 300 / config.num_samples
config.test_split_proportion = 200 / config.num_samples

# モデルは AAN + フローエンコーダ/デコーダ（軽量）
config.model = 'action-angle-network'
config.encoder_decoder_type = 'flow'
config.latent_size = 32
config.activation = 'sigmoid'
config.flow_type = 'shear'
config.num_flow_layers = 4
config.num_angular_velocity_net_layers = 2
config.polar_action_angles = True

# 学習ハイパラ（軽量）
config.learning_rate = 1e-3
config.batch_size = 128
config.num_train_steps = 1000
config.eval_cadence = 100

# 正則化（論文相当の項を弱めに使用）
config.regularizations = ml_collections.ConfigDict({
    'actions': 1.0,
    'angular_velocities': 0.0,
    'encoded_decoded_differences': 0.0,
})

# 乱数キーと時刻
rng = jax.random.PRNGKey(config.rng_seed)
times = jnp.arange(config.num_samples) * config.time_delta

# シミュレーションパラメータをサンプリング
rng, srng = jax.random.split(rng)
sim_params = hsim.sample_simulation_parameters(config.simulation_parameter_ranges.to_dict(),
                                              config.num_trajectories, srng)

# (q, p) を生成（[T, D]）
q_all, p_all = train.get_generate_canonical_coordinates_fn(config)(times, sim_params)
q_all = np.asarray(q_all); p_all = np.asarray(p_all)
q_all.shape, p_all.shape

In [ ]:
# 生成データの一部を位相空間で可視化
_ = hsim.static_plot_coordinates_in_phase_space(q_all[:300], p_all[:300],
                                               title='Harmonic: sample (first 300 steps)')
plt.show()

## 2) 学習の実行（学習パイプラインを利用）
`train.train_and_evaluate` をそのまま使って学習・評価・保存を行います。
学習・評価済みの内容は workdir にチェックポイント・メトリクス・設定が保存されます。

In [ ]:
# 作業ディレクトリを用意
workdir = './tmp/end_to_end_harmonic_aan'
pathlib.Path(workdir).mkdir(parents=True, exist_ok=True)

# 学習＆評価（数分〜）
scaler, best_state, aux = train.train_and_evaluate(config, workdir)
print('Training done. Saved to:', workdir)

## 3) 学習結果の読み込みと可視化
- 1ステップ予測（jump=10 など）を真値と並べて位相空間に描画
- Hamiltonian の推移（真値と予測）を比較

In [ ]:
# 保存物の読み込み（設定・スケーラ・ベストモデル・補助データ）
loaded_config, loaded_scaler, loaded_state, loaded_aux = analysis.load_from_workdir(workdir)
loaded_config

In [ ]:
# 1ステップ予測（jump=10）を取得し、真値と並べて描画
jump = 10
true_q, true_p, true_H = analysis.get_test_trajectories(workdir, jump)
pred_q, pred_p, pred_H = analysis.get_one_step_predicted_trajectories(workdir, jump)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), frameon=False)
_ = hsim.static_plot_coordinates_in_phase_space(true_q, true_p, title=f'True (jump={jump})', ax=axes[0])
_ = hsim.static_plot_coordinates_in_phase_space(pred_q, pred_p, title=f'Predicted (jump={jump})', ax=axes[1])
plt.show()

# Hamiltonian の比較（予測 vs 真値）
plt.figure(figsize=(6,3))
plt.plot(true_H, label='H true')
plt.plot(pred_H, label='H predicted', alpha=0.8)
plt.xlabel('time index')
plt.ylabel('Hamiltonian')
plt.legend()
plt.title('Hamiltonian over time (test pairs)')
plt.tight_layout()
plt.show()

## 4) 連鎖（再帰）多ステップ予測の可視化
1ステップ予測器を繰り返し適用して、多ステップ先まで予測した結果の位相空間プロットを確認します。

In [ ]:
multi_q, multi_p, multi_H = analysis.get_recursive_multi_step_predicted_trajectories(workdir, jump=10)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), frameon=False)
_ = hsim.static_plot_coordinates_in_phase_space(true_q, true_p, title='True (for reference)', ax=axes[0])
_ = hsim.static_plot_coordinates_in_phase_space(multi_q, multi_p, title='Recursive multi-step (jump=10)', ax=axes[1])
plt.show()

plt.figure(figsize=(6,3))
plt.plot(true_H, label='H true')
plt.plot(multi_H, label='H recursive', alpha=0.8)
plt.xlabel('time index')
plt.ylabel('Hamiltonian')
plt.legend()
plt.title('Hamiltonian over time (recursive multi-step)')
plt.tight_layout()
plt.show()

## 参考: モデルの切替
- Neural ODEにしたい場合: `config.model = 'neural-ode'`、`config.encoder_decoder_type = 'mlp'` 等へ変更。
- Hamiltonian Neural Networkにしたい場合: `config.model = 'hamiltonian-neural-network'`。
- MLPエンコーダ/デコーダへ切り替えたい場合: `config.encoder_decoder_type = 'mlp'`。

必要に応じて `action_angle_networks/configs/**` を参照し、層数や活性化関数などを調整してください。